# Does a deeper, structurally-informed `ProfileMLP` exploit more diverse data better?

`diversity_sweep.ipynb` (`analysis of parameters for viability/`) found that as library diversity
`d0` grows at fixed sequencing budget (`D=1e8` there), top-1000 recovery declines for BOTH the raw
protocol signal and the `ProfileMLP` -- but not in perfect lockstep: the MLP's recovery
crosses back above the raw protocol's at the very largest `d0` tested (`1,000,000`), and its
Pearson `r` crosses over much earlier (~`50,000`-`100,000`). That's a real but modest and
LATE denoising effect. This notebook asks: can a bigger, more structurally-informed model
capture that effect earlier and more strongly -- i.e. actually benefit from the extra data a
more diverse library provides, instead of mostly just absorbing noisier labels?

**Two architectures, trained on IDENTICAL data at every `d0`** (only the model changes):

- **`ShallowProfileMLP`** -- the plain `Linear->BatchNorm->Dropout->GELU` x2 + scalar head used
  everywhere else in this project. `~26K` parameters.
- **`DeepProfileMLP`** -- new, this notebook. `~90K` parameters (3.4x), and structurally informed
  by the TRUE generative model (`F` additive + `J` pairwise), Deep-Sets style:
  - a shared per-position embedding (`Linear(20 -> 16) + GELU`, applied identically to each of
    the `L=7` positions -- same weights reused across positions, like a learned, nonlinear `F`);
  - **average-pooled** over the 7 positions -> a permutation-invariant "profile" summary;
  - a **pairwise branch**: every one of the `21` unordered position pairs `(i,j)` gets its two
    embeddings concatenated and passed through a small shared MLP, then **average-pooled** over
    the 21 pairs -> an epistasis-aware summary (a strictly more expressive analogue of
    `MLP_bilinear_head_anticorrelated.ipynb`'s `BilinearHead`, which computed a raw bilinear form
    directly on one-hot inputs and gave only marginal improvement -- see that notebook's Verdict
    section);
  - both pooled summaries concatenated with the raw flattened one-hot (140-d) and fed into a
    **4-hidden-layer** dense head (`256->128->64->32`, vs. the shallow model's 2 layers
    `128->64`) -- meaningfully deeper, with genuinely more capacity to exploit a larger, more
    diverse training set before saturating.

Same `mu=10`, `rho=1e-4`, `D=5e8` (FIXED here, not adaptive -- `diversity_sweep.ipynb` itself used
`D=1e8`), `noise_viab=3` (NOT `0.5` like `diversity_sweep.ipynb` -- an unresolved discrepancy,
flagged but not changed here), and its OWN `DIVERSITY_GRID` (`200` to `200,000`, a superset of
`diversity_sweep.ipynb`'s `5,000`-`200,000` range at the small end) -- results are comparable on
the shared `5,000`-`200,000` overlap, not point-for-point identical grids. `N0=150*N1` here too
(same wet-lab ratio constraint, see `diversity_sweep.ipynb` section 1). Both models use the SAME adaptive
`batch_size` scheme (`max(256, n_train // 50)`) at every `d0` -- deliberately, to isolate the
architecture as the only variable; `diversity_sweep.ipynb`'s own training-diagnostics section
already flags the batch-size/steps-per-epoch question as a separate, orthogonal thing to test.

## 0. Setup

In [ ]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
# Must run before the first `import jax` anywhere -- same convention as the sibling notebooks.

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

from flax import nnx
import optax

from sequence_classesV1 import *
from analysisV1 import *
from initialize_weights import load_F_viab_aav9_mlp, load_J_viab_aav9_mlp, NUM_AMINO_ACIDS, NUM_POSITIONS

print(f"JAX backend: {jax.default_backend()} -- devices: {jax.devices()}")

## 1. Ground truth weights

Identical real AAV9 `F_viab`/`J_viab` and permuted `F_sel`/`J_sel` as `diversity_sweep.ipynb`
(`F_sel`/`J_sel` are constructed but never used below -- only the viability channel, `target1`,
is swept here). No single fixed `sequences` pool -- each `d0` gets its own inside the sweep loop
(section 5).

In [ ]:
F_viab = load_F_viab_aav9_mlp()
J_viab = load_J_viab_aav9_mlp()

key_F, key_J = jax.random.split(jax.random.key(0), 2)
sigma_F = jax.random.permutation(key_F, NUM_AMINO_ACIDS)
F_sel   = F_viab[sigma_F, :]
sigma_J = jax.random.permutation(key_J, NUM_AMINO_ACIDS)
J_sel   = J_viab[:, :, sigma_J, :][:, :, :, sigma_J]

print(f"F_viab shape: {F_viab.shape}   J_viab shape: {J_viab.shape}")

## 2. Fixed 50,000-sequence evaluation pool (SAME key in every notebook)

Section 6's own train/test split shrinks WITH `d0` -- at the smallest grid point (`d0=200`), the
test fold is only `100` sequences, so `topk_recovery(..., k=1000)` silently clamps to
`k=min(1000, 100)=100` and becomes the WHOLE test fold: a trivially perfect, meaningless
"recovery". More generally, comparing recovery numbers ACROSS different `d0` isn't quite
apples-to-apples even when it's not degenerate, since the population being ranked (the test
fold) is a different, differently-sized set at every grid point.

Fix: a SEPARATE, FIXED `50,000`-sequence pool, generated once with a hardcoded key
(`EVAL_POOL_KEY_SEED=999`, `EVAL_POOL_SIZE=50_000` -- reuse this EXACT pair of values in any
other notebook that should be directly comparable to this one) and held out from training
entirely -- it plays no role in section 6's per-`d0` pool/train/val/test split. Every model
trained at every `d0` gets evaluated on this SAME `50,000` sequences (section 9), so top-1000
recovery becomes comparable both across `d0` within this notebook AND across any other notebook
using the same fixed key, independent of how big or small the training pool itself is.

In [ ]:
EVAL_POOL_KEY_SEED = 999   # fixed across every notebook -- reuse this exact value for comparable numbers
EVAL_POOL_SIZE     = 50_000

def compute_score_array(seq, F, J, L=NUM_POSITIONS):
    """Noiseless ground-truth score, decoupled from any Protocol instance -- same formula as
    Protocol.compute_score, applied directly to an arbitrary sequence array."""
    scores = jnp.sum(F[seq, jnp.arange(L)], axis=1)
    for i in range(L):
        for j in range(i + 1, L):
            scores = scores + J[i, j, seq[:, i], seq[:, j]]
    return scores

eval_sequences = jax.random.randint(jax.random.key(EVAL_POOL_KEY_SEED),
                                     shape=(EVAL_POOL_SIZE, NUM_POSITIONS),
                                     minval=0, maxval=NUM_AMINO_ACIDS)
eval_viab_score = np.asarray(compute_score_array(eval_sequences, F_viab, J_viab))
X_eval = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[np.asarray(eval_sequences)].reshape(EVAL_POOL_SIZE, -1)

print(f"fixed evaluation pool: {EVAL_POOL_SIZE:,} sequences (key seed={EVAL_POOL_KEY_SEED})")
print(f"GT score range on eval pool: [{eval_viab_score.min():.2f}, {eval_viab_score.max():.2f}]")

## 3. Fixed `mu=10`, `rho=1e-4`, `D=5e8`, `T_viab=0.8`, `noise_viab=0.5`, and the `d0` grid

Same fixed `D` (not adaptive) as `diversity_sweep.ipynb`, but NOT the same `noise_viab` (`3` here
vs `0.5` there) -- so the "protocol" and "shallow MLP" curves here are NOT expected to reproduce
that notebook's numbers point-for-point; treat this section as this notebook's own internal
reference (shallow vs deep), not a cross-notebook sanity check.

In [ ]:
RHO_FIXED        = 1e-4
D_FIXED          = 5e8
T_VIAB_FIXED     = 0.8
NOISE_VIAB_FIXED = 3
MU_FIXED         = 10
K_TOPK           = 1000
eps              = 1.0

DIVERSITY_GRID = [200, 1_000, 5_000, 10_000, 20_000, 50_000, 100_000, 200_000]

print(f"mu={MU_FIXED}  rho={RHO_FIXED}  D={D_FIXED:.0e}  T_viab={T_VIAB_FIXED}  noise_viab={NOISE_VIAB_FIXED}")
print(f"diversity (d0) grid: {DIVERSITY_GRID}")
for d0_val in DIVERSITY_GRID:
    print(f"  d0={d0_val:>9,}  ->  N1={MU_FIXED * d0_val / RHO_FIXED:.3e}  "
          f"reads/sequence (D/d0) = {D_FIXED / d0_val:.1f}")

## 4. Simulate the SAME experimental configuration on the fixed eval pool

To get a genuine `GT<->protocol` recovery number on the fixed `50,000`-sequence eval pool
(section 2), we run a SEPARATE, independent `ProtocolV3` simulation dedicated to it alone --
NOT mixed into any `d0`-sized training pool (that would dilute `D`/`d0` and distort `mu` at
every sweep point, since the eval pool's `50,000` sequences would then dominate small-`d0`
points). Same `mu=10`/`rho`/`D`/`T_viab`/`noise_viab` as the rest of this notebook, just with
`N1` recomputed for the eval pool's OWN `d0=50,000` (`N1_eval = mu * 50,000 / rho`) so it
experiences the identical `mu`. Since none of these parameters depend on the training sweep's
`d0`, this simulation runs ONCE, not once per grid point -- `target1_eval` (and everything
derived from it) is a FIXED reference the whole sweep can be compared against.

In [ ]:
N1_eval = MU_FIXED * EVAL_POOL_SIZE / RHO_FIXED

protocol_eval = ProtocolV3(multinomialNGS=True, N0=N1_eval*150, N1=N1_eval,
        dilution_factor=10, sequences=eval_sequences, D=D_FIXED,
        F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
        noise_viab=NOISE_VIAB_FIXED, noise_sel=0.5, T_sel=1, T_viab=T_VIAB_FIXED,
        )
protocol_eval._rho = float(RHO_FIXED)

_bio_row_eval, ngs_row_eval = protocol_eval.loop_DE()
lambda0p_eval, lambda2p_eval, _lambda3p_eval = (np.asarray(a) for a in ngs_row_eval)
target1_eval = np.log((lambda2p_eval + eps) / (lambda0p_eval + eps))

r_gt_protocol_eval   = pearson(eval_viab_score, target1_eval)
rec_gt_protocol_eval = topk_recovery(eval_viab_score, target1_eval, k=K_TOPK)
mse_gt_protocol_eval = float(np.mean((eval_viab_score - target1_eval) ** 2))

print(f"mu (eval pool) = {protocol_eval._rho * protocol_eval.N1 / EVAL_POOL_SIZE:.2f}")
print(f"GT <-> protocol on the fixed eval pool: r={r_gt_protocol_eval:.3f}  "
      f"top-{K_TOPK} recovery={100 * rec_gt_protocol_eval:.1f}%  MSE={mse_gt_protocol_eval:.3f}")

## 5. Two architectures: `ShallowProfileMLP` (baseline) vs `DeepProfileMLP` (new)

`ShallowProfileMLP` is verbatim this project's standard `ProfileMLP` (renamed here only to make
every plot/table unambiguous about which model is which).

`DeepProfileMLP`: per-position embedding (shared across the `L=7` positions) -> average-pooled
profile branch + average-pooled pairwise branch (all `21` unordered position pairs) -> concatenated
with the raw one-hot -> 4-hidden-layer dense head. `nnx.List(...)` is required here (unlike the
shallow model's named `linear1`/`linear2` attributes) because the head's layer count is a
parameter (`hidden_dims`), not hardcoded.

In [ ]:
class ShallowProfileMLP(nnx.Module):
    """
    This project's standard ProfileMLP (Linear + BatchNorm + Dropout + gelu, twice, then a
    scalar linear head) over the 140-d one-hot input -- unchanged, just renamed for clarity.
    """

    def __init__(self, input_dim: int, hidden_dims: tuple[int, int] = (128, 64),
                 dropout_rate: float = 0.1, *, rngs: nnx.Rngs):
        h1, h2 = hidden_dims
        self.linear1    = nnx.Linear(input_dim, h1, rngs=rngs)
        self.batchnorm1 = nnx.BatchNorm(h1, use_running_average=False, rngs=rngs)
        self.dropout1   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear2    = nnx.Linear(h1, h2, rngs=rngs)
        self.batchnorm2 = nnx.BatchNorm(h2, use_running_average=False, rngs=rngs)
        self.dropout2   = nnx.Dropout(rate=dropout_rate, rngs=rngs)
        self.linear3    = nnx.Linear(h2, 1, rngs=rngs)

    def __call__(self, x, train: bool, rngs: nnx.Rngs = None):
        x = self.linear1(x)
        self.batchnorm1.use_running_average = not train
        x = self.batchnorm1(x)
        x = nnx.gelu(x)
        x = self.dropout1(x, rngs=rngs) if train else x
        x = self.linear2(x)
        self.batchnorm2.use_running_average = not train
        x = self.batchnorm2(x)
        x = nnx.gelu(x)
        x = self.dropout2(x, rngs=rngs) if train else x
        x = self.linear3(x)
        return x[:, 0]

In [ ]:
class PositionEmbed(nnx.Module):
    """Shared per-position embedding: maps each position's 20-d one-hot to a d_emb-d learned
    embedding via ONE Linear+GELU, with the SAME weights applied to all L=7 positions (parameter
    sharing across position -- a Deep-Sets/PointNet-style per-element encoder)."""

    def __init__(self, A: int, d_emb: int, *, rngs: nnx.Rngs):
        self.linear = nnx.Linear(A, d_emb, rngs=rngs)

    def __call__(self, x_pos):  # (batch, L, A)
        return nnx.gelu(self.linear(x_pos))  # (batch, L, d_emb)


class PairwiseHead(nnx.Module):
    """For every one of the L*(L-1)/2 unordered position pairs (i,j): concatenate the two
    positions' embeddings, pass through a small shared 2-layer MLP, then AVERAGE-POOL over all
    pairs. A strictly more expressive analogue of MLP_bilinear_head_anticorrelated.ipynb's
    BilinearHead (which used a raw bilinear form on one-hot inputs directly, x_i^T W x_j, and
    gave only marginal improvement there) -- here the interaction is learned through a nonlinear
    MLP over LEARNED embeddings, not a fixed bilinear form over raw indicators."""

    def __init__(self, L: int, d_emb: int, d_pair: int, *, rngs: nnx.Rngs):
        self.pairs_i = jnp.array([i for i in range(L) for j in range(i + 1, L)])
        self.pairs_j = jnp.array([j for i in range(L) for j in range(i + 1, L)])
        self.linear1 = nnx.Linear(2 * d_emb, 2 * d_pair, rngs=rngs)
        self.linear2 = nnx.Linear(2 * d_pair, d_pair, rngs=rngs)

    def __call__(self, emb):  # emb: (batch, L, d_emb)
        e_i = emb[:, self.pairs_i, :]  # (batch, n_pairs, d_emb)
        e_j = emb[:, self.pairs_j, :]
        pair_feat = jnp.concatenate([e_i, e_j], axis=-1)  # (batch, n_pairs, 2*d_emb)
        h = nnx.gelu(self.linear1(pair_feat))
        h = self.linear2(h)  # (batch, n_pairs, d_pair)
        return jnp.mean(h, axis=1)  # average pool over the 21 pairs -> (batch, d_pair)


class DeepProfileMLP(nnx.Module):
    """
    Per-position embedding -> [average-pooled profile branch, average-pooled pairwise branch,
    raw one-hot] concatenated -> a 4-hidden-layer dense head (deeper than ShallowProfileMLP's 2).
    """

    def __init__(self, L: int, A: int, d_emb: int = 16, d_pair: int = 16,
                 hidden_dims: tuple = (256, 128, 64, 32), dropout_rate: float = 0.1,
                 *, rngs: nnx.Rngs):
        self.L, self.A = L, A
        self.pos_embed = PositionEmbed(A, d_emb, rngs=rngs)
        self.pairwise  = PairwiseHead(L, d_emb, d_pair, rngs=rngs)

        input_dim = d_emb + d_pair + L * A
        dims = (input_dim,) + hidden_dims
        n_layers = len(hidden_dims)
        self.linears    = nnx.List([nnx.Linear(dims[i], dims[i + 1], rngs=rngs) for i in range(n_layers)])
        self.batchnorms = nnx.List([nnx.BatchNorm(dims[i + 1], use_running_average=False, rngs=rngs) for i in range(n_layers)])
        self.dropouts   = nnx.List([nnx.Dropout(rate=dropout_rate, rngs=rngs) for _ in range(n_layers)])
        self.out        = nnx.Linear(hidden_dims[-1], 1, rngs=rngs)

    def __call__(self, x_flat, train: bool, rngs: nnx.Rngs = None):
        batch = x_flat.shape[0]
        x_pos = x_flat.reshape(batch, self.L, self.A)

        emb             = self.pos_embed(x_pos)     # (batch, L, d_emb)
        profile_pooled  = jnp.mean(emb, axis=1)      # (batch, d_emb)  -- avg pool over positions
        pairwise_pooled = self.pairwise(emb)         # (batch, d_pair) -- avg pool over pairs

        h = jnp.concatenate([profile_pooled, pairwise_pooled, x_flat], axis=-1)
        for linear, bn, drop in zip(self.linears, self.batchnorms, self.dropouts):
            h = linear(h)
            bn.use_running_average = not train
            h = bn(h)
            h = nnx.gelu(h)
            h = drop(h, rngs=rngs) if train else h
        return self.out(h)[:, 0]


n_params_shallow = sum(p.size for p in jax.tree.leaves(
    nnx.state(ShallowProfileMLP(input_dim=NUM_POSITIONS * NUM_AMINO_ACIDS, rngs=nnx.Rngs(0)), nnx.Param)))
n_params_deep = sum(p.size for p in jax.tree.leaves(
    nnx.state(DeepProfileMLP(L=NUM_POSITIONS, A=NUM_AMINO_ACIDS, rngs=nnx.Rngs(0)), nnx.Param)))
print(f"ShallowProfileMLP: {n_params_shallow:,} params")
print(f"DeepProfileMLP:    {n_params_deep:,} params  ({n_params_deep / n_params_shallow:.1f}x)")

## 6. Generic trainer: same recipe for both architectures

`train_step`/`eval_step`/`predict_log_enrichment`/`train_model` are architecture-agnostic --
`model_ctor(rngs)` builds whichever model is being trained, so the exact same warmup-cosine-decay
AdamW / early-stopping loop (identical to every other sweep in this project) is reused for both
`ShallowProfileMLP` and `DeepProfileMLP`.

In [ ]:
def split_train_val(X, y, val_frac=0.15, seed=0):
    rng   = np.random.default_rng(seed)
    idx   = rng.permutation(len(X))
    n_val = int(len(X) * val_frac)
    val_idx, train_idx = idx[:n_val], idx[n_val:]
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]


@nnx.jit
def train_step(model, optimizer, x, y, rngs):
    def loss_fn(model, rngs):
        y_pred = model(x, train=True, rngs=rngs)
        return jnp.mean((y_pred - y) ** 2)
    loss, grads = nnx.value_and_grad(loss_fn)(model, rngs)
    optimizer.update(model, grads)
    return loss


@nnx.jit
def eval_step(model, x, y):
    y_pred = model(x, train=False)
    return jnp.mean((y_pred - y) ** 2)


@nnx.jit
def predict_log_enrichment(model, x):
    return model(x, train=False)


def train_model(model_ctor, X_train, y_train, X_val, y_val, epochs=300, batch_size=256,
                 peak_lr=1e-3, final_lr=1e-5, weight_decay=0, patience=20, seed=0, verbose=True):
    """model_ctor(rngs) -> a fresh nnx.Module (ShallowProfileMLP or DeepProfileMLP instance)."""
    rngs  = nnx.Rngs(seed)
    model = model_ctor(rngs)

    n_train         = X_train.shape[0]
    steps_per_epoch = max(n_train // batch_size, 1)
    total_steps     = steps_per_epoch * epochs

    lr_schedule_fn = optax.warmup_cosine_decay_schedule(
        init_value=0., peak_value=peak_lr,
        warmup_steps=int(total_steps * 0.1),
        decay_steps=int(total_steps * 0.9),
        end_value=final_lr,
    )
    optimizer = nnx.Optimizer(
        model, optax.adamw(learning_rate=lr_schedule_fn, weight_decay=weight_decay), wrt=nnx.Param
    )

    X_train, y_train = jnp.asarray(X_train), jnp.asarray(y_train)
    X_val,   y_val   = jnp.asarray(X_val),   jnp.asarray(y_val)

    shuffle_key = jax.random.key(seed)
    best_val, best_state, bad_epochs, best_epoch, final_epoch = float("inf"), None, 0, 0, 0

    for epoch in range(epochs):
        shuffle_key, perm_key = jax.random.split(shuffle_key)
        perm = jax.random.permutation(perm_key, n_train)
        Xs, ys = X_train[perm], y_train[perm]

        for start in range(0, n_train - n_train % batch_size, batch_size):
            xb = Xs[start:start + batch_size]
            yb = ys[start:start + batch_size]
            train_step(model, optimizer, xb, yb, rngs)

        val_loss = float(eval_step(model, X_val, y_val))
        final_epoch = epoch
        if val_loss < best_val - 1e-6:
            best_val, bad_epochs, best_epoch = val_loss, 0, epoch
            best_state = nnx.state(model)
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"  early stop at epoch {epoch}, best val MSE = {best_val:.4f}")
                break

    if best_state is not None:
        nnx.update(model, best_state)
    return model, best_val, best_epoch, final_epoch, steps_per_epoch

## 7. Sweep: one fresh pool + one fresh `ProtocolV3` per `d0`, both architectures trained on IDENTICAL data

For each `d0`: draw a fresh pool, build `N1 = mu * d0 / rho`, construct a fresh `ProtocolV3`
(fixed `D=D_FIXED`), run one `loop_DE()` round, split 50/50. Then train `ShallowProfileMLP` AND
`DeepProfileMLP` on the EXACT same `(Xtr, ytr, Xva, yva)` (same `batch_size`, same `seed=0`) so
any difference in the results is attributable to the architecture alone.

Also re-simulates the fixed eval pool a SECOND way at every `d0_val`: `protocol_eval_matched`,
sequenced at `D_eval_matched = (D_FIXED / d0_val) * EVAL_POOL_SIZE` -- the SAME reads/sequence
this `d0_val`'s own training pool gets, rather than section 4's fixed `D_FIXED/EVAL_POOL_SIZE`
depth. This is the fair benchmark: "if you just sequenced the eval pool at the depth THIS
training run actually had, what would raw NGS alone give you" -- since `GT<->protocol` on
section 4's fixed-depth pool is a constant ~93% regardless of `d0`, while the MLP's OWN training
labels get shallower (and noisier) as `d0` grows, comparing the MLP against that fixed ~93%
ceiling was never quite apples-to-apples.

In [ ]:
records = []
base_key = jax.random.key(1)

for d0_val in tqdm(DIVERSITY_GRID, desc="diversity (d0) sweep -- shallow vs deep"):
    pool_key  = jax.random.fold_in(base_key, d0_val)
    sequences = jax.random.randint(pool_key, shape=(d0_val, NUM_POSITIONS),
                                    minval=0, maxval=NUM_AMINO_ACIDS)
    N1_val = MU_FIXED * d0_val / RHO_FIXED

    protocol = ProtocolV3(multinomialNGS=True, N0=N1_val*150, N1=N1_val,
            dilution_factor=10, sequences=sequences, D=D_FIXED,
            F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
            noise_viab=NOISE_VIAB_FIXED, noise_sel=0.5, T_sel=1, T_viab=T_VIAB_FIXED,
            )
    protocol._rho = float(RHO_FIXED)
    viab_score = np.array(protocol.compute_score(F_viab, J_viab))

    _bio_row, ngs_row = protocol.loop_DE()
    lambda0p, lambda2p, _lambda3p = (np.asarray(a) for a in ngs_row)
    target1 = np.log((lambda2p + eps) / (lambda0p + eps))

    # Matched-depth protocol: same fixed eval pool (section 2), but sequenced at the SAME
    # reads/sequence (D_FIXED / d0_val) as THIS d0's training pool, instead of section 4's
    # fixed D_FIXED/EVAL_POOL_SIZE depth -- the fair benchmark for "does the MLP beat what raw
    # NGS alone would give at the depth it was actually trained on". Re-simulated every d0_val
    # since the matched depth changes with it (unlike section 4's protocol_eval, computed once).
    D_eval_matched = (D_FIXED / d0_val) * EVAL_POOL_SIZE
    protocol_eval_matched = ProtocolV3(multinomialNGS=True, N0=N1_eval*150, N1=N1_eval,
            dilution_factor=10, sequences=eval_sequences, D=D_eval_matched,
            F_viab=F_viab, J_viab=J_viab, F_sel=F_sel, J_sel=J_sel,
            noise_viab=NOISE_VIAB_FIXED, noise_sel=0.5, T_sel=1, T_viab=T_VIAB_FIXED,
            )
    protocol_eval_matched._rho = float(RHO_FIXED)
    _bio_row_em, ngs_row_em = protocol_eval_matched.loop_DE()
    lambda0p_em, lambda2p_em, _lambda3p_em = (np.asarray(a) for a in ngs_row_em)
    target1_eval_matched = np.log((lambda2p_em + eps) / (lambda0p_em + eps))

    X_all = np.eye(NUM_AMINO_ACIDS, dtype=np.float32)[np.asarray(sequences)].reshape(d0_val, -1)
    idx_train, idx_test = train_test_split(np.arange(d0_val), test_size=0.5, random_state=0)
    X_train_full, X_test   = X_all[idx_train], X_all[idx_test]
    viab_score_test        = viab_score[idx_test]
    target1_train, target1_test = target1[idx_train], target1[idx_test]

    Xtr, ytr, Xva, yva = split_train_val(X_train_full, target1_train, val_frac=0.15, seed=0)
    batch_size = max(256, len(Xtr) // 50)

    rec = dict(d0=d0_val, batch_size=batch_size,
               r_gt_protocol=pearson(viab_score_test, target1_test),
               rec_gt_protocol=topk_recovery(viab_score_test, target1_test, k=K_TOPK),
               D_eval_matched=D_eval_matched,
               r_gt_protocol_eval_matched=pearson(eval_viab_score, target1_eval_matched),
               rec_gt_protocol_eval_matched=topk_recovery(eval_viab_score, target1_eval_matched, k=K_TOPK),
               mse_gt_protocol_eval_matched=float(np.mean((eval_viab_score - target1_eval_matched) ** 2)))

    model_ctors = {
        "shallow": lambda rngs: ShallowProfileMLP(input_dim=NUM_POSITIONS * NUM_AMINO_ACIDS, rngs=rngs),
        "deep":    lambda rngs: DeepProfileMLP(L=NUM_POSITIONS, A=NUM_AMINO_ACIDS, rngs=rngs),
    }

    for arch_name, ctor in model_ctors.items():
        model, val_mse, best_epoch, final_epoch, steps_per_epoch = train_model(
            ctor, Xtr, ytr, Xva, yva, epochs=300, patience=20, batch_size=batch_size, verbose=False)
        pred_test = np.asarray(predict_log_enrichment(model, jnp.asarray(X_test)))
        pred_eval = np.asarray(predict_log_enrichment(model, jnp.asarray(X_eval)))

        rec[f"r_gt_mlp_{arch_name}"]       = pearson(viab_score_test, pred_test)
        rec[f"r_protocol_mlp_{arch_name}"] = pearson(target1_test, pred_test)
        rec[f"rec_gt_mlp_{arch_name}"]       = topk_recovery(viab_score_test, pred_test, k=K_TOPK)
        rec[f"rec_protocol_mlp_{arch_name}"] = topk_recovery(target1_test, pred_test, k=K_TOPK)
        # fixed 50,000-sequence eval pool (section 2) -- comparable across d0 and across notebooks,
        # unlike rec_gt_mlp_* above whose population is the in-sweep test fold (shrinks with d0)
        rec[f"rec_gt_mlp_eval_{arch_name}"]       = topk_recovery(eval_viab_score, pred_eval, k=K_TOPK)
        rec[f"r_protocol_mlp_eval_{arch_name}"]   = pearson(target1_eval, pred_eval)
        rec[f"rec_protocol_mlp_eval_{arch_name}"] = topk_recovery(target1_eval, pred_eval, k=K_TOPK)
        rec[f"r_protocol_matched_mlp_{arch_name}"]   = pearson(target1_eval_matched, pred_eval)
        rec[f"rec_protocol_matched_mlp_{arch_name}"] = topk_recovery(target1_eval_matched, pred_eval, k=K_TOPK)
        rec[f"mse_gt_mlp_eval_{arch_name}"] = float(np.mean((eval_viab_score - pred_eval) ** 2))
        rec[f"val_mse_{arch_name}"]     = val_mse
        rec[f"best_epoch_{arch_name}"]  = best_epoch
        rec[f"total_steps_{arch_name}"] = steps_per_epoch * (final_epoch + 1)

    records.append(rec)

arch_comparison_df = pd.DataFrame(records)
arch_comparison_df

## 8. Pearson correlation vs `d0`: shallow vs deep

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(arch_comparison_df["d0"], arch_comparison_df["r_gt_protocol"],    "o-", color="gray",   label="GT <-> protocol")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["r_gt_mlp_shallow"], "o-", color="tab:blue", label="GT <-> MLP (shallow)")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["r_gt_mlp_deep"],    "o-", color="tab:red",  label="GT <-> MLP (deep)")
ax.set_xscale("log")
ax.set_ylim(0, 1)
ax.set_xlabel("d0 (library diversity)")
ax.set_ylabel("Pearson r (on the log enrichments)")
ax.set_title(f"Pearson r vs d0 -- mu={MU_FIXED}, rho={RHO_FIXED:g}, D={D_FIXED:.0e} (fixed), "
             f"T_viab={T_VIAB_FIXED}, noise_viab={NOISE_VIAB_FIXED}")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

## 9. Top-1000 recovery on the FIXED 50,000-sequence eval pool: shallow vs deep

Same `50,000` sequences (section 2) at every `d0` -- comparable across the WHOLE grid, including
the smallest `d0` values (`200`, `1,000`), unlike a train/test split carved out of the
`d0`-sized training pool itself. TWO `GT<->protocol` references: the dashed gray line is section
4's fixed-depth simulation (constant, `D_FIXED/EVAL_POOL_SIZE` reads/sequence always); the solid
dark-gray curve is section 7's MATCHED-depth simulation, re-sequenced at the SAME `D_FIXED/d0_val`
reads/sequence the training pool got at that point -- the fairer benchmark for "does the MLP
actually beat what raw NGS alone gives at the depth it was trained on".

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axhline(100 * rec_gt_protocol_eval, color="gray", lw=1.5, ls="--",
           label=f"GT <-> protocol, fixed depth (r={r_gt_protocol_eval:.3f})")
ax.plot(arch_comparison_df["d0"], 100 * arch_comparison_df["rec_gt_protocol_eval_matched"], "o-",
        color="dimgray", label="GT <-> protocol, MATCHED depth (D/d0)")
ax.plot(arch_comparison_df["d0"], 100 * arch_comparison_df["rec_gt_mlp_eval_shallow"], "o-", color="tab:blue", label="GT <-> MLP (shallow)")
ax.plot(arch_comparison_df["d0"], 100 * arch_comparison_df["rec_gt_mlp_eval_deep"],    "o-", color="tab:red",  label="GT <-> MLP (deep)")
ax.axhline(100 * K_TOPK / EVAL_POOL_SIZE, color="black", lw=1, ls=":", label="random baseline")
ax.set_xscale("log")
ax.set_ylim(0, 100)
ax.set_xlabel("d0 (training pool size)")
ax.set_ylabel(f"Top-{K_TOPK} recovery on the fixed {EVAL_POOL_SIZE:,}-sequence eval pool (%)")
ax.set_title("GT <-> MLP recovery on a COMMON, fixed evaluation population -- shallow vs deep")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

print(arch_comparison_df[["d0", "rec_gt_protocol_eval_matched", "rec_gt_mlp_eval_shallow", "rec_gt_mlp_eval_deep"]])

## 10. MSE on the FIXED 50,000-sequence eval pool: protocol vs shallow vs deep

Same population as sections 4/7/9, but MSE instead of Pearson `r` or top-1000 recovery --
Pearson `r` already showed the fixed-vs-matched-depth protocol curves clearly diverging
(`r=0.989` at `d0=10,000` down to `r=0.953` at `d0=200,000`) even though top-1000 recovery barely
moved (~92% throughout, section 9) -- MSE is the metric the MLP is ACTUALLY trained to minimize
(`train_step`'s loss function, section 6), so it's the most direct "is the model doing its job"
check, and should track the `r` story rather than the recovery one.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.axhline(mse_gt_protocol_eval, color="gray", lw=1.5, ls="--",
           label=f"GT <-> protocol, fixed depth (MSE={mse_gt_protocol_eval:.2f})")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["mse_gt_protocol_eval_matched"], "o-",
        color="dimgray", label="GT <-> protocol, MATCHED depth (D/d0)")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["mse_gt_mlp_eval_shallow"], "o-", color="tab:blue", label="GT <-> MLP (shallow)")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["mse_gt_mlp_eval_deep"],    "o-", color="tab:red",  label="GT <-> MLP (deep)")
ax.set_xscale("log")
ax.set_xlabel("d0 (training pool size)")
ax.set_ylabel(f"MSE vs GT on the fixed {EVAL_POOL_SIZE:,}-sequence eval pool")
ax.set_title("MSE(GT, prediction) on a COMMON, fixed evaluation population -- shallow vs deep")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)
fig.tight_layout()
plt.show()

print(arch_comparison_df[["d0", "mse_gt_protocol_eval_matched", "mse_gt_mlp_eval_shallow", "mse_gt_mlp_eval_deep"]])

## 11. Training diagnostics: does the deeper model actually use more of the extra data?

Same diagnostic as `diversity_sweep.ipynb` section 10, side by side for both architectures. If
`total_steps_deep` grows with `d0` about as little as `total_steps_shallow` does (both flat,
since they share the same `batch_size` schedule), any improvement from the deep model has to
come from its extra CAPACITY/structure per step, not from training longer -- worth knowing
before concluding the deeper model "uses more data better" in a training-dynamics sense.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(arch_comparison_df["d0"], arch_comparison_df["best_epoch_shallow"], "o-", color="tab:blue", label="shallow")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["best_epoch_deep"],    "o-", color="tab:red",  label="deep")
ax.set_xscale("log")
ax.set_xlabel("d0")
ax.set_ylabel("epoch of best val MSE")
ax.set_title("Convergence speed")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)

ax = axes[1]
ax.plot(arch_comparison_df["d0"], arch_comparison_df["total_steps_shallow"], "o-", color="tab:blue", label="shallow")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["total_steps_deep"],    "o-", color="tab:red",  label="deep")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("d0")
ax.set_ylabel("total optimizer steps taken")
ax.set_title("Total gradient updates")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)

ax = axes[2]
ax.plot(arch_comparison_df["d0"], arch_comparison_df["val_mse_shallow"], "o-", color="tab:blue", label="shallow")
ax.plot(arch_comparison_df["d0"], arch_comparison_df["val_mse_deep"],    "o-", color="tab:red",  label="deep")
ax.set_xscale("log")
ax.set_xlabel("d0")
ax.set_ylabel("best validation MSE")
ax.set_title("Own-distribution fit quality")
ax.legend()
ax.grid(True, linestyle="--", alpha=0.3)

fig.tight_layout()
plt.show()

print(arch_comparison_df[["d0", "batch_size",
                           "best_epoch_shallow", "best_epoch_deep",
                           "total_steps_shallow", "total_steps_deep",
                           "val_mse_shallow", "val_mse_deep"]])

## How to read this

- **Sanity check first**: `GT <-> protocol` (fixed depth) and `GT <-> MLP (shallow)` here should
  closely match `diversity_sweep.ipynb`'s own numbers (same fixed `D`, same grid, same shallow
  architecture) -- if they don't, something about this notebook's setup diverged and the "deep"
  comparison isn't trustworthy yet.
- **Top-1000 recovery (section 9) barely moves for the matched-depth protocol curve (~92%
  throughout, even at `d0=200,000`'s `2,500` reads/sequence) -- use section 8 (Pearson `r`) or
  section 10 (MSE) to actually see the depth effect.** Recovery only asks "are the top 2% of
  50,000 sequences correctly identified", which turns out to be fairly robust to the noise levels
  in this grid; `r` and MSE look at the WHOLE distribution and decline/rise clearly
  (`r`: `0.989` at `d0=10,000` down to `0.953` at `d0=200,000`). Don't read section 9 alone as
  "depth doesn't matter" -- it means recovery specifically is an insensitive metric here, not
  that the underlying signal quality is unaffected.
- **Section 10 (MSE) is the most direct check, since MSE is literally what `train_step` minimizes**
  (section 6): does either MLP's `mse_gt_mlp_eval_*` curve drop BELOW the matched-depth protocol's
  `mse_gt_protocol_eval_matched` at any `d0`? That is unambiguous evidence the model is a better
  estimator of GT than a single raw NGS measurement at the same depth -- a cleaner signal than
  recovery for "is the MLP actually helping."
- **Does either MLP curve (section 8, 10) separate from the matched-depth protocol MORE, or
  EARLIER (smaller `d0`), for deep vs shallow?** That's the direct answer to "does a bigger,
  structurally-informed model exploit diversity better."
- **If the deep model does NOT improve on the shallow one**, section 11's diagnostics matter: if
  `total_steps_deep` is basically identical to `total_steps_shallow` (both flat across `d0`,
  since batch size is shared), the deeper model may simply not be getting enough gradient steps
  to make use of its extra capacity at large `d0` -- worth then testing a fixed, smaller
  `batch_size` (steps/epoch scaling WITH `d0`) as a follow-up, isolating "more steps" as its own
  variable rather than conflating it with "more capacity" here.
- The pairwise/profile branches' average pooling is what makes this architecture MORE sensitive,
  in principle, to a richer training distribution: pooling forces the model to learn signals that
  generalize ACROSS positions/pairs rather than memorizing per-sequence quirks, which should
  matter more as the pool gets bigger and more varied -- if it doesn't show up here, that's a
  meaningful negative result too, not just a null one.